# SLM Pretraining on TF1 — English Fable Generator

Train a **small language model from scratch** on the TF1 fable dataset to show that a tiny model (~10M / ~30M params) can still write coherent children's fables.

**How to run (Colab):** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
Artifacts (GGUF + Modelfile) are written to **Google Drive** so they survive a runtime recycle.

**Pipeline:** mount Drive -> setup & data -> hyperparameters -> dataset (loss-masked) -> model + WSD training -> train 10M & 30M -> export to Ollama.

In [ ]:
# --- Mount Google Drive (so trained GGUFs persist across runtime recycles) ---
from google.colab import drive
drive.mount('/content/drive')                 # click "Authorize" in the popup
import os
DRIVE = "/content/drive/MyDrive/slm_tf1"       # all outputs land here on your Drive
os.makedirs(DRIVE, exist_ok=True)
print("Drive mounted:", os.path.isdir("/content/drive/MyDrive"))

## Step 1 - Setup & data

Clone the repo, install dependencies, then build the training corpus + tokenizer (idempotent - skips if already present).

- **Corpus:** 150k TF1 fables. Each example is formatted as `conditioning (the 5 narrative slots) \n <|story|> fable <|end|>`.
- **Tokenizer:** a small custom **BPE (vocab 12k)** trained on the fables - keeps a tiny model's embedding table small.

In [ ]:
# --- Clone repo + install deps + build corpus & tokenizer (run once) ---
import os, subprocess
if not os.path.exists("/content/tinystory-vn"):
    subprocess.run("git clone -q https://github.com/tungd/tinystory-vn.git", shell=True, cwd="/content")
os.chdir("/content/tinystory-vn")
subprocess.run("git checkout -q feat/slm-pretrain-tf1", shell=True)
subprocess.run('pip -q install "datasets>=2.20" "tokenizers>=0.19" "transformers>=4.44" torch accelerate', shell=True)

if not os.path.exists("data/tf1/tokenizer.json"):
    subprocess.run("python -m trieulh.scripts.prepare_tf1_pretrain --train-n 600000 --min-words 60 --max-words 320 --test-n 400 --out data/tf1", shell=True)
    subprocess.run("python -m trieulh.scripts.train_tokenizer --data data/tf1/train.jsonl --out data/tf1/tokenizer.json --vocab-size 12000", shell=True)
print("data ready:", sum(1 for _ in open("data/tf1/train.jsonl")), "training rows")

## Step 2 - Training hyperparameters

The **"method" of the training** - every knob you can tune, and what it does. Edit here; the cells below read these values.

In [ ]:
# --- Export both models to GGUF + Ollama Modelfile, saved to Drive ---import os, subprocessif not os.path.exists("llama.cpp"):    subprocess.run("git clone -q https://github.com/ggerganov/llama.cpp && pip -q install -r llama.cpp/requirements.txt", shell=True)# FIX 2: teach llama.cpp that our custom ByteLevel BPE = gpt-2 pre-tokenizer (else it errors)CHK = "43ed72f253f7c15ca73d6505923aebbe171a253149da589154dd3977604534bc"bp  = "llama.cpp/conversion/base.py"; L = open(bp).read().split("\n")if not any("gpt-2" in x and CHK in x for x in L):    for i, ln in enumerate(L):        if 'raise NotImplementedError("BPE pre-tokenizer was not recognized' in ln:            ind = ln[:len(ln) - len(ln.lstrip())]            L[i] = f'{ind}if chkhsh == "{CHK}": return "gpt-2"\n{ln}'; break    open(bp, "w").write("\n".join(L))# TEMPLATE = training format exactly: user prompt, newline, story separator (no chat/system markup)TMPL = 'TEMPLATE """{{ .Prompt }}\\n<|story|>"""\n'# Guard: ensure model directories existfor s in ["10M", "30M"]:    assert os.path.isdir(f"{DRIVE}/{s}"), f"missing model dir {DRIVE}/{s} — train it first"    tag = f"slm-{s.lower()}"    subprocess.run(f"python llama.cpp/convert_hf_to_gguf.py {DRIVE}/{s} --outfile {DRIVE}/{tag}.gguf --outtype q8_0", shell=True)    open(f"{DRIVE}/Modelfile-{s}", "w").write(        f"FROM ./{tag}.gguf\n{TMPL}"        'PARAMETER temperature 0.8\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.3\n'        'PARAMETER stop "<|end|>"\nPARAMETER num_ctx 512\n')print("exported to Drive:", sorted(os.listdir(DRIVE)))# --- Export 10M-distilled to GGUF + Modelfile (same TEMPLATE and params) ---dist_tag = "slm-10m-distilled"subprocess.run(f"python llama.cpp/convert_hf_to_gguf.py {DRIVE}/10M-distilled "               f"--outfile {DRIVE}/{dist_tag}.gguf --outtype q8_0", shell=True)open(f"{DRIVE}/Modelfile-10M-distilled", "w").write(    f"FROM ./{dist_tag}.gguf\n{TMPL}"    "PARAMETER temperature 0.8\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.3\n"    "PARAMETER stop \"<|end|>\"\nPARAMETER num_ctx 512\n")print("10M-distilled GGUF exported ->", f"{DRIVE}/{dist_tag}.gguf")print("updated Drive:", sorted(os.listdir(DRIVE)))

## Step 3 - Dataset with loss masking

Tokenize each example and **mask the conditioning prefix** (labels = -100) so the model predicts only the **story**, never the prompt. `cond_len` (a character offset written by the data prep step) is converted to a **token** count before masking. The collator pads a batch, marking padding with label -100 / attention_mask 0.

In [ ]:
# --- Build the tokenized, loss-masked dataset + a padding collator ---
import json, torch
from transformers import (LlamaConfig, LlamaForCausalLM, PreTrainedTokenizerFast,
                          Trainer, TrainingArguments)

tok = PreTrainedTokenizerFast(tokenizer_file="data/tf1/tokenizer.json",
        unk_token="<|unk|>", pad_token="<|pad|>", eos_token="<|end|>")

def encode(row):
    ids = tok(row["text"], truncation=True, max_length=SEQ_LEN)["input_ids"]
    # cond_len is a CHARACTER offset -> re-tokenize the prefix to get its TOKEN count
    n_cond = min(len(tok(row["text"][:row["cond_len"]])["input_ids"]), len(ids))
    labels = [-100] * n_cond + ids[n_cond:]     # mask prompt tokens; learn only the story
    return {"input_ids": ids, "labels": labels}

print("encoding..."); DS = [encode(json.loads(l)) for l in open("data/tf1/train.jsonl")]
print("encoded", len(DS), "examples")

def collator(features):
    pad = tok.pad_token_id
    m = max(len(f["input_ids"]) for f in features)
    fill = lambda seq, val: seq + [val] * (m - len(seq))
    return {
        "input_ids":      torch.tensor([fill(f["input_ids"], pad)  for f in features]),
        "labels":         torch.tensor([fill(f["labels"], -100)    for f in features]),
        "attention_mask": torch.tensor([[1]*len(f["input_ids"]) + [0]*(m-len(f["input_ids"])) for f in features]),
    }

## Step 4 - Model + WSD training loop

`train(size)` builds a **Llama-style model from random init** (`tie_word_embeddings=True` shares input/output embeddings) and trains with **AdamW** + a **Warmup-Stable-Decay** LR schedule wired through `LambdaLR`. Reads every value from the hyperparameters cell.

In [ ]:
# --- Train one model size from scratch (uses the hyperparameters above) ---
def train(size):
    cfg = LlamaConfig(vocab_size=tok.vocab_size, max_position_embeddings=SEQ_LEN,
                      tie_word_embeddings=True, **ARCH[size])
    model = LlamaForCausalLM(cfg)
    print(f"[{size}] params(M):", round(sum(p.numel() for p in model.parameters())/1e6, 1))

    optimizer = torch.optim.AdamW(model.parameters(), lr=PEAK_LR,
                                  betas=ADAM_BETAS, weight_decay=WEIGHT_DECAY)

    def wsd(step):                                  # Warmup-Stable-Decay multiplier in [0,1]
        warm, dec = int(WARMUP_FRAC*STEPS), int(DECAY_FRAC*STEPS)
        if step < warm:          return step / max(1, warm)               # 0 -> 1  (warmup)
        if step > STEPS - dec:   return max(0.0, (STEPS-step)/max(1,dec))  # 1 -> 0  (decay)
        return 1.0                                                         # hold peak (stable)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, wsd)

    # NOTE: on resume, the WSD LambdaLR step counter restarts from 0 (HF Trainer does not
    # persist LambdaLR state). The LR ramps up briefly again before re-entering the stable
    # plateau — acceptable for this project; loss continues to fall.
    out_dir = f"{DRIVE}/{size}"                      # DRIVE = /content/drive/MyDrive/slm_tf1
    os.makedirs(out_dir, exist_ok=True)
    args = TrainingArguments(
        output_dir=out_dir, max_steps=STEPS, fp16=FP16,
        per_device_train_batch_size=BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUM,
        max_grad_norm=GRAD_CLIP, logging_steps=LOG_EVERY,
        save_steps=SAVE_EVERY, save_total_limit=3,
        lr_scheduler_type="constant", report_to=[],   # WSD applied via LambdaLR
    )
    trainer = Trainer(model=model, args=args, train_dataset=DS, data_collator=collator,
                      optimizers=(optimizer, scheduler))
    ckpts = [d for d in os.listdir(out_dir) if d.startswith("checkpoint-")] if os.path.isdir(out_dir) else []
    trainer.train(resume_from_checkpoint=bool(ckpts))
    model.save_pretrained(out_dir); tok.save_pretrained(out_dir)
    # transformers 5.x saves tokenizer_class="TokenizersBackend" which llama.cpp/AutoTokenizer
    # cannot load -> rewrite it to the loadable PreTrainedTokenizerFast.
    p = f"{out_dir}/tokenizer_config.json"; c = json.load(open(p))
    c["tokenizer_class"] = "PreTrainedTokenizerFast"; json.dump(c, open(p, "w"))
    print(f"[{size}] saved -> {out_dir}")

## Step 5 - Train 30M (full Chinchilla)

v2: trains **30M only** to ~600M tokens (~7 900 optimizer steps, effective batch 256).  
This takes **several hours on a T4** — Drive checkpoints every 400 steps make it **resumable across sessions**: just re-run this cell and training picks up from the latest checkpoint.

The 10M teacher-distilled model is produced in a later task (distillation from the 30M teacher).

Watch the live **Step | Training Loss** table (updates every `LOG_EVERY` steps).  
Loss should fall from ~6 down to ~1.6–1.8 over the full run.

In [ ]:
train("30M")   # v2: full Chinchilla ~600M tokens; resumable via Drive checkpoints

## Step 6b - Distill 30M teacher → 10M student (token-level KD)

Load the frozen **30M teacher** and train a fresh **10M student** with `distillation_loss` (α=0.5, T=2.0): a blend of KL-divergence on softened logits and hard cross-entropy on story tokens only.

- **KD_STEPS ≈ 2 600** — ~200M tokens (~1.5 epochs over the 600k subset)
- fp16 GradScaler, WSD schedule, grad-clip applied identically to the 30M run
- Saved to `{DRIVE}/10M-distilled` with `tokenizer_class` rewritten for compatibility

In [ ]:
# === Distill 30M teacher -> 10M student (token-level KD) ===
import torch
from trieulh.scripts.distill import distillation_loss
from transformers import LlamaConfig, LlamaForCausalLM

teacher = LlamaForCausalLM.from_pretrained(f"{DRIVE}/30M").to("cuda").eval()
for p in teacher.parameters(): p.requires_grad_(False)

student = LlamaForCausalLM(LlamaConfig(
    vocab_size=tok.vocab_size, max_position_embeddings=SEQ_LEN,
    tie_word_embeddings=True, **ARCH["10M"])).to("cuda")
print("student params(M):", round(sum(p.numel() for p in student.parameters())/1e6, 1))

KD_STEPS = 2600           # ~200M tokens (~1.5 epochs over the 600k subset)
opt = torch.optim.AdamW(student.parameters(), lr=3e-3, betas=ADAM_BETAS, weight_decay=WEIGHT_DECAY)
def wsd(s):
    w, d = int(0.02*KD_STEPS), int(0.2*KD_STEPS)
    return s/max(1,w) if s < w else (max(0.,(KD_STEPS-s)/max(1,d)) if s > KD_STEPS-d else 1.)
sched = torch.optim.lr_scheduler.LambdaLR(opt, wsd)

from torch.utils.data import DataLoader
loader = DataLoader(DS, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
student.train(); step = 0
scaler = torch.cuda.amp.GradScaler(enabled=FP16)
while step < KD_STEPS:
    for batch in loader:
        batch = {k: v.to("cuda") for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=FP16):
            with torch.no_grad():
                t_logits = teacher(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            s_logits = student(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            # shift for next-token prediction; flatten
            sl = s_logits[:, :-1].reshape(-1, s_logits.size(-1))
            tl = t_logits[:, :-1].reshape(-1, t_logits.size(-1))
            yl = batch["labels"][:, 1:].reshape(-1)
            loss = distillation_loss(sl, tl, yl, T=2.0, alpha=0.5)
        opt.zero_grad(); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
        scaler.step(opt); scaler.update(); sched.step(); step += 1
        if step % LOG_EVERY == 0: print(f"kd step {step}/{KD_STEPS} loss {loss.item():.3f}")
        if step >= KD_STEPS: break

out_dir = f"{DRIVE}/10M-distilled"; import os; os.makedirs(out_dir, exist_ok=True)
student.save_pretrained(out_dir); tok.save_pretrained(out_dir)
import json as J; p=f"{out_dir}/tokenizer_config.json"; c=J.load(open(p)); c["tokenizer_class"]="PreTrainedTokenizerFast"; J.dump(c, open(p,"w"))
print("distilled student saved ->", out_dir)

## Step 6 - Export to Ollama (GGUF)

Convert each model to **GGUF (q8)** and write an Ollama `Modelfile` whose `TEMPLATE` reproduces the exact training format - `prompt + "\n<|story|>"`, with **no chat wrapper / no system role** (the from-scratch model never saw one). Two fixes are needed for a custom-BPE tiny model:
1. **tokenizer_class** rewrite (done in `train`) so the converter can read the tokenizer.
2. **llama.cpp pre-tokenizer patch** - register our BPE hash as `gpt-2`-style (ByteLevel), which the converter otherwise rejects.

Outputs land in your Drive folder: `slm-10m.gguf`, `slm-30m.gguf`, `Modelfile-10M`, `Modelfile-30M`.

In [ ]:
# --- Export both models to GGUF + Ollama Modelfile, saved to Drive ---import os, subprocessif not os.path.exists("llama.cpp"):    subprocess.run("git clone -q https://github.com/ggerganov/llama.cpp && pip -q install -r llama.cpp/requirements.txt", shell=True)# FIX 2: teach llama.cpp that our custom ByteLevel BPE = gpt-2 pre-tokenizer (else it errors)CHK = "43ed72f253f7c15ca73d6505923aebbe171a253149da589154dd3977604534bc"bp  = "llama.cpp/conversion/base.py"; L = open(bp).read().split("\n")if not any("gpt-2" in x and CHK in x for x in L):    for i, ln in enumerate(L):        if 'raise NotImplementedError("BPE pre-tokenizer was not recognized' in ln:            ind = ln[:len(ln) - len(ln.lstrip())]            L[i] = f'{ind}if chkhsh == "{CHK}": return "gpt-2"\n{ln}'; break    open(bp, "w").write("\n".join(L))# TEMPLATE = training format exactly: user prompt, newline, story separator (no chat/system markup)TMPL = 'TEMPLATE """{{ .Prompt }}\\n<|story|>"""\n'for s in ["10M", "30M"]:    assert os.path.isdir(f"{DRIVE}/{s}"), f"missing model dir {DRIVE}/{s} — train it first"    tag = f"slm-{s.lower()}"    subprocess.run(f"python llama.cpp/convert_hf_to_gguf.py {DRIVE}/{s} --outfile {DRIVE}/{tag}.gguf --outtype q8_0", shell=True)    open(f"{DRIVE}/Modelfile-{s}", "w").write(        f"FROM ./{tag}.gguf\n{TMPL}"        'PARAMETER temperature 0.8\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.3\n'        'PARAMETER stop "<|end|>"\nPARAMETER num_ctx 512\n')print("exported to Drive:", sorted(os.listdir(DRIVE)))# --- Export 10M-distilled to GGUF + Modelfile (same TEMPLATE and params) ---dist_tag = "slm-10m-distilled"subprocess.run(f"python llama.cpp/convert_hf_to_gguf.py {DRIVE}/10M-distilled "               f"--outfile {DRIVE}/{dist_tag}.gguf --outtype q8_0", shell=True)open(f"{DRIVE}/Modelfile-10M-distilled", "w").write(    f"FROM ./{dist_tag}.gguf\n{TMPL}"    "PARAMETER temperature 0.8\nPARAMETER top_p 0.9\nPARAMETER repeat_penalty 1.3\n"    "PARAMETER stop \"<|end|>\"\nPARAMETER num_ctx 512\n")print("10M-distilled GGUF exported ->", f"{DRIVE}/{dist_tag}.gguf")print("updated Drive:", sorted(os.listdir(DRIVE)))